In [0]:
# Imports
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    when,
    date_format,
    current_timestamp
)

print("Imports successful")

Imports successful


In [0]:
# Build Gold claims summary
def build_claims_summary():
    
    # Read from Silver claims
    silver_claims = spark.sql("""
        SELECT *
        FROM silver.claims
        WHERE is_active = true
    """)
    
    # Aggregate by provider and month
    gold_df = silver_claims \
        .withColumn(
            "claim_month",
            date_format(col("updated_at"), "yyyy-MM")
        ) \
        .groupBy("provider_id", "claim_month") \
        .agg(
            count("claim_id").alias("total_claims"),
            sum("claim_amount").alias("total_amount"),
            count(
                when(col("claim_status") == "APPROVED", 1)
            ).alias("approved_claims"),
            count(
                when(col("claim_status") == "DENIED", 1)
            ).alias("denied_claims"),
            avg("claim_amount").alias("avg_claim_amount")
        ) \
        .withColumn("_updated_at", current_timestamp())
    
    return gold_df

In [0]:
# Write Gold claims summary
def write_to_gold(gold_df):
    
    # Overwrite Gold table completely
    gold_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("gold.claims_summary")
    
    print(f"Gold table updated with {gold_df.count()} rows")

In [0]:
# Run Silver to Gold pipeline
print("Starting Silver to Gold pipeline")
print("=" * 50)

# Step 1 - Build aggregations
gold_df = build_claims_summary()

# Step 2 - Write to Gold
write_to_gold(gold_df)

print("=" * 50)
print("Silver to Gold pipeline complete")

Starting Silver to Gold pipeline
Gold table updated with 3 rows
Silver to Gold pipeline complete


In [0]:
# Check Gold table
print("=== GOLD CLAIMS SUMMARY ===")
spark.sql("SELECT * FROM gold.claims_summary").show()

=== GOLD CLAIMS SUMMARY ===
+-----------+-----------+------------+------------+---------------+-------------+------------------+--------------------+
|provider_id|claim_month|total_claims|total_amount|approved_claims|denied_claims|  avg_claim_amount|         _updated_at|
+-----------+-----------+------------+------------+---------------+-------------+------------------+--------------------+
|      PRV01|    2024-01|           3|      5800.0|              1|            0|1933.3333333333333|2026-09-13 00:44:...|
|      PRV03|    2024-01|           2|      6700.0|              1|            1|            3350.0|2026-09-13 00:44:...|
|      PRV02|    2024-01|           3|      5050.0|              0|            1|1683.3333333333333|2026-09-13 00:44:...|
+-----------+-----------+------------+------------+---------------+-------------+------------------+--------------------+

